In [1]:
import os
import openai
import duckdb
import pandas as pd
import time

duckdb.sql("""
           CREATE OR REPLACE TABLE component AS
           SELECT "ch.chat_id" AS chat_id
           FROM read_csv('analysis_data/chats_graph/secondcomponent_id_107635.csv')
           """)

duckdb.sql("""
           CREATE OR REPLACE TABLE chats_all AS
           SELECT chat_id, text_content
           FROM read_csv('preprocess_data/chat_nodes.csv')
           """)

duckdb.sql("""
           CREATE OR REPLACE TABLE chats AS
           SELECT b.chat_id as chat_id, b.text_content as text_content
           FROM component as a, chats_all as b
           WHERE a.chat_id == b.chat_id
           """)

In [2]:
client = openai.OpenAI(api_key=os.environ.get("MARITACA_API_KEY"),
                       base_url="https://chat.maritaca.ai/api")

prompt_system = """Você é um classificador de desinformação. Analise o texto fornecido e classifique o assunto principal na melhor categoria. Sua resposta deve conter o número de uma das categorias, somente o número.

1. Política e Eleições: Fraude em urnas, ataque a instituições públicas.
2. Economia: Impostos, inflação, crise, PIX, Bolsa Família, financiamento.
3. Saúde e Ciência: Vacinas, tratamentos, alimentação, crise sanitária, pseudociências, ceticismo sobre ciência.
4. Segurança e Golpes: Link de prêmio, alarmismo sobre crimes e golpes.
5. Sociedade: Preconceitos, discriminação de minorias, pessoas e etnias.
6. Calamidades: Invasão, apocalipse, guerra.
7. Indeterminado: apenas se não houver relação com algum assunto anterior.
"""

In [3]:
def get_response(string):
    for attempt in range(1,6):
        try:
            response = client.chat.completions.create(
                model="sabia-4",
                messages=[
                    {"role": "system", "content": prompt_system},
                    {"role": "user", "content": string}
                ],
                stream=False,
                tools=[],
                max_tokens=5,
                temperature=0.0
            )
            return response
        except openai.RateLimitError:
            print("HTTP 429, attempt", attempt)
            time.sleep(2*attempt)
    raise Exception

def run_api(classification, df):
    i = 1
    for string in df['text_content']:
        try:
            res = get_response(string)
        except Exception:
            print("Stopping. Failed at index", i)
            return
        classification.append(res.choices[0].message.content)
        time.sleep(1.0)
        i += 1

In [4]:
df = duckdb.sql("SELECT * FROM chats").df()
classification = []
run_api(classification, df)